# 🔍 벡터 검색 (Vector Search) 실습

Azure AI Search의 벡터 검색 기능을 실습합니다.

## 📋 학습 내용

1. **기본 벡터 검색**: 의미 기반 검색
2. **벡터 검색 vs 키워드 검색**: 차이점 비교
3. **하이브리드 검색**: 키워드 + 벡터 결합
4. **필터링 벡터 검색**: 벡터 검색 + 필터 조건
5. **멀티 벡터 검색**: 여러 필드를 동시에 검색

## 🔍 주요 개념

### 벡터 검색이란?
- **의미 기반 검색**: 키워드가 정확히 일치하지 않아도 의미가 유사하면 검색
- **임베딩 벡터**: 텍스트를 고차원 벡터로 변환
- **코사인 유사도**: 벡터 간 유사도를 측정하여 관련성 판단

### 키워드 검색 vs 벡터 검색

| 구분 | 키워드 검색 | 벡터 검색 |
|------|-------------|-----------|
| 방식 | 단어 일치 | 의미 유사도 |
| 예시 | "노트북" → "노트북" | "노트북" → "랩탑", "컴퓨터" |
| 알고리즘 | BM25 | HNSW + Cosine |
| 장점 | 정확한 매칭 | 자연스러운 검색 |
| 단점 | 유사어 못 찾음 | 계산 비용 높음 |

### 하이브리드 검색
- 키워드 검색(BM25)과 벡터 검색을 결합
- 두 방식의 장점을 모두 활용
- Reciprocal Rank Fusion(RRF)으로 점수 통합

---

## 1️⃣ 라이브러리 임포트 및 초기화

In [1]:
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI
from dotenv import load_dotenv
import os

# 환경 변수 로드
load_dotenv(override=True)

# Azure AI Search 설정
search_endpoint = os.getenv("SEARCH_ENDPOINT")
search_key = os.getenv("SEARCH_ADMIN_KEY")
index_name = os.getenv("SEARCH_INDEX_NAME")

# Azure OpenAI 설정
openai_endpoint = os.getenv("AZURE_OPEN_AI_ENDPOINT")
openai_key = os.getenv("AZURE_OPEN_AI_KEY")
embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_EMBEDDING_API_VERSION")

# SearchClient 초기화
search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=index_name,
    credential=AzureKeyCredential(search_key)
)

# OpenAI Client 초기화 (키 우선, AOAI_AUTH=aad 또는 키 없으면 AAD 폴백)
_use_key = (
    os.getenv("AOAI_AUTH", "").lower() != "aad"
    and openai_key and openai_key.strip() and not openai_key.startswith("your-")
)
if _use_key:
    openai_client = AzureOpenAI(
        azure_endpoint=openai_endpoint,
        api_key=openai_key,
        api_version=api_version,
    )
    _auth_mode = "API Key"
else:
    aoai_token_provider = get_bearer_token_provider(
        DefaultAzureCredential(),
        "https://cognitiveservices.azure.com/.default",
    )
    openai_client = AzureOpenAI(
        azure_endpoint=openai_endpoint,
        azure_ad_token_provider=aoai_token_provider,
        api_version=api_version,
    )
    _auth_mode = "Entra ID (AAD)"

print("✅ 클라이언트 초기화 완료")
print(f"   Endpoint: {search_endpoint}")
print(f"   Index: {index_name}")
print(f"   OpenAI Auth: {_auth_mode}")


✅ 클라이언트 초기화 완료
   Endpoint: https://ai-search-foundry-iq-cj.search.windows.net
   Index: products-index
   OpenAI Auth: Entra ID (AAD)


## 2️⃣ 유틸리티 함수 정의

검색에 사용할 헬퍼 함수들을 정의합니다.

In [2]:
def get_embedding(text):
    """
    텍스트를 벡터로 변환
    """
    response = openai_client.embeddings.create(
        input=text,
        model=embedding_deployment
    )
    return response.data[0].embedding

def print_results(results, title="검색 결과", show_score=True):
    """
    검색 결과를 보기 좋게 출력
    """
    print(f"\n{'='*80}")
    print(f"{title}")
    print('='*80)
    
    result_list = list(results)
    if not result_list:
        print("검색 결과가 없습니다.")
        return
    
    for idx, result in enumerate(result_list, 1):
        if show_score and '@search.score' in result:
            score = result['@search.score']
            print(f"\n{idx}. [점수: {score:.4f}] {result.get('title', 'N/A')}")
        else:
            print(f"\n{idx}. {result.get('title', 'N/A')}")
        
        print(f"   브랜드: {result.get('brand', 'N/A')} | 카테고리: {result.get('category', 'N/A')}")
        print(f"   가격: {result.get('normal_price', 0):,}원")
        
        if 'review' in result and result['review']:
            review = result['review'][:100]
            print(f"   리뷰: {review}...")

print("✅ 유틸리티 함수 정의 완료")

✅ 유틸리티 함수 정의 완료


---

## 3️⃣ 벡터 검색 알고리즘 이해 (개념 학습)

### 🔍 K-Nearest Neighbors (KNN) 문제란?

**KNN은 "K개의 가장 가까운 이웃을 찾는 문제"입니다.**
- "운동화"와 가장 유사한 제품 10개 찾기
- 모든 벡터 검색은 본질적으로 KNN 문제입니다

이 KNN 문제를 푸는 방법에는 크게 두 가지가 있습니다:

---

### 📊 KNN 문제를 푸는 두 가지 방법

#### 1. **Exhaustive Search (전수 탐색)** 
```
모든 벡터와 하나하나 거리를 계산
```
- **100% 정확**: 모든 벡터를 비교하므로 절대 빠뜨리지 않음
- **느린 속도**: 데이터가 N개면 N번 계산 필요 (O(N))
- **메모리 효율**: 추가 자료구조 불필요
- **Azure 구성**: `ExhaustiveKnnAlgorithmConfiguration` + `knn-profile`

**예시**: 제품 247개라면 247번 모두 계산

---

#### 2. **HNSW (Hierarchical Navigable Small World)** 
```
그래프 구조로 빠르게 근사 탐색
```
- **95-99% 정확**: 일부 벡터만 탐색 (근사 알고리즘)
- **빠른 속도**: 그래프 경로를 따라 탐색 (O(log N))
- **높은 메모리**: 그래프 구조 저장 필요
- **Azure 구성**: `HnswAlgorithmConfiguration` + `hnsw-profile`

**예시**: 제품 247개 중 ~50개만 계산하고 최적해 근사

---

### 📊 비교표

| 구분 | Exhaustive Search | HNSW (ANN) |
|------|-------------------|------------|
| 정식 명칭 | Exact KNN (Flat) | Approximate NN (HNSW) |
| 검색 방식 | 모든 벡터 거리 계산 | 그래프 기반 근사 탐색 |
| 정확도 | 100% (Exact) | High recall (parameter-dependent) |
| 시간 특성 | O(N) | Sub-linear (empirical) |
| 메모리 특성 | 인덱스 구조 단순, 런타임 비용 큼 | 그래프 인덱스 메모리 사용 |
| 적합한 규모 | 소규모 / 검증용 | 중·대규모 / 실서비스 |


---

### ⚠️ 이 노트북의 제약사항

**실제로는 두 알고리즘을 직접 비교할 수 없습니다!**

이유:
1. Azure AI Search에서 **벡터 필드는 하나의 프로필만 지정 가능**
2. 현재 `content_vector` 필드는 `hnsw-profile`로 설정됨
3. 아래 코드는 둘 다 같은 필드를 사용 → **둘 다 HNSW로 검색됨**
4. 진짜 비교를 하려면:
   - `content_vector_exhaustive` (knn-profile 사용)
   - `content_vector_hnsw` (hnsw-profile 사용)
   - 두 개의 별도 필드 필요

**따라서 아래는 개념 학습용 코드입니다.**
- 실제로는 같은 알고리즘(HNSW)으로 두 번 검색
- 결과와 속도가 거의 동일하게 나옴
- 대규모 데이터(수만 개 이상)에서는 차이가 명확함

---

### 💡 실무 선택 가이드

| 데이터 규모 | 권장 알고리즘 | 이유 |
|-------------|--------------|------|
| **< 1만 개** | 둘 다 OK | 속도 차이 미미 |
| **1만 ~ 10만** | HNSW 권장 | 체감 속도 향상 |
| **> 10만 개** | HNSW 필수 | Exhaustive는 너무 느림 |

**대부분의 실무에서는 HNSW 사용합니다.**

---

## 4️⃣ 벡터 검색 알고리즘 실습 (개념 학습)


In [3]:
# 실용적인 벡터 검색 사례 테스트
# content_vector는 제품명 + 브랜드 + 카테고리를 마크다운 형태로 포함

print("\n" + "="*80)
print("💡 벡터 검색의 강점: 의미 기반 검색")
print("="*80)
print("content_vector는 다음 정보를 포함합니다:")
print("   - 제품명 (title)")
print("   - 브랜드 (brand)")
print("   - 카테고리 (category)")
print("이를 통해 더욱 풍부한 의미 기반 검색이 가능합니다.\n")

test_queries = [
    ("여름에 입을 가벼운 옷", "자연어 질문 → 패션 제품 검색"),
    ("아이 장난감", "간단한 키워드 → 카테고리 기반 검색"),
    ("나이키 신발", "브랜드 + 제품 → 브랜드 정보 활용"),
    ("고급 화장품", "추상적 표현 → 의미 이해")
]

for query, description in test_queries:
    print(f"\n{'='*80}")
    print(f"🔍 검색어: '{query}'")
    print(f"📝 사례: {description}")
    print('='*80)
    
    query_vector = get_embedding(query)
    
    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=3,
        fields="content_vector"  # 제품명 + 브랜드 + 카테고리 포함
    )
    
    results = search_client.search(
        search_text=None,
        vector_queries=[vector_query],
        select=["title", "brand", "category", "normal_price", "review"],
        top=3
    )
    
    for idx, result in enumerate(results, 1):
        score = result.get('@search.score', 0)
        print(f"{idx}. [점수: {score:.4f}] {result['title']}")
        print(f"   {result['brand']} | {result['category']} | 가격: {result['normal_price']:,}원")
        print(f"   {result}")


💡 벡터 검색의 강점: 의미 기반 검색
content_vector는 다음 정보를 포함합니다:
   - 제품명 (title)
   - 브랜드 (brand)
   - 카테고리 (category)
이를 통해 더욱 풍부한 의미 기반 검색이 가능합니다.


🔍 검색어: '여름에 입을 가벼운 옷'
📝 사례: 자연어 질문 → 패션 제품 검색


1. [점수: 0.6252] 노스페이스 NJ3LR05 남성 아이스 페이스 자켓 (여름 경량 바람막이 자켓)
   노스페이스 | 스포츠/레져 | 가격: 65,550원
   {'normal_price': 65550, 'review': '노스페이스 아이스 페이스 자켓은 여름철 가볍게 입기 딱 좋은 바람막이예요. 소재가 얇고 통기성이 좋아서 땀 차는 느낌 없이 시원하게 착용할 수 있었습니다. 특히 바람을 잘 막아줘서 아침저녁 선선할 때도 부담 없이 입기 좋았고, 휴대하기 편한 경량 디자인 덕분에 여행 갈 때도 항상 챙기게 되네요. 색상도 무난해서 어디에나 잘 어울립니다. 가격 대비 품질이 만족스러워 추천합니다.', 'brand': '노스페이스', 'title': '노스페이스 NJ3LR05 남성 아이스 페이스 자켓 (여름 경량 바람막이 자켓)', 'category': '스포츠/레져', '@search.score': 0.62516624, '@search.reranker_score': None, '@search.highlights': None, '@search.captions': None}
2. [점수: 0.6039] 앤더슨벨 커트 스트라이프 롱슬리브 티셔츠 atb1247m(ECRU)
   앤더슨벨 | 패션의류 | 가격: 103,500원
   {'normal_price': 103500, 'review': '앤더슨벨 커트 스트라이프 롱슬리브 티셔츠는 정말 기대 이상이에요. 부드러운 에크루 컬러와 세련된 스트라이프 패턴이 일상복으로 딱 좋고, 신축성 좋은 소재 덕분에 착용감도 편안합니다. 특히 소매 끝부분의 커팅 디테일이 독특해서 은근히 포인트가 되네요. 가격대가 조금 있지만 디자인과 품질 모두 만족스러워서 잘 산 것 같아요. 봄, 가을 간절기용으로 적극 추천합니다.', 'brand': '앤더슨벨', 'title': '앤더슨벨 커트 스트라이프 롱슬리브 티셔츠 atb1247m(ECRU)', 'category': '패션의류', '@search.score': 0.60388434, 

1. [점수: 0.6713] 5315-실바니안 그로서리 마켓
   토이트론 | 유아동 | 가격: 70,000원
   {'normal_price': 70000, 'review': '실바니안 그로서리 마켓은 아이가 정말 좋아하는 장난감이에요. 다양한 식료품 모형과 작은 진열대가 있어 아이가 가게 놀이를 하면서 상상력이 많이 자라더라고요. 부품이 튼튼하고 마감도 깔끔해서 오래 사용할 수 있을 것 같아요. 가격대가 조금 있지만 아이 교육 장난감으로는 만족스러운 선택이었습니다. 아이와 함께 놀면서 소근육 발달에도 도움 되는 것 같아 추천합니다.', 'brand': '토이트론', 'title': '5315-실바니안 그로서리 마켓', 'category': '유아동', '@search.score': 0.6712999, '@search.reranker_score': None, '@search.highlights': None, '@search.captions': None}
2. [점수: 0.6659] 5096-실바니안 어린이 병원(2815)
   토이트론 | 유아동 | 가격: 100,000원
   {'normal_price': 100000, 'review': '아이에게 생일 선물로 5096-실바니안 어린이 병원을 구매했는데 정말 만족스러워요. 병원 건물과 소품들이 디테일하게 잘 만들어져 있어서 아이가 역할놀이를 하며 집중하는 모습이 보기 좋았습니다. 작은 인형들과 함께 다양한 상황을 연출할 수 있어 상상력 발달에 도움도 되는 것 같아요. 내구성도 튼튼해서 오래 사용할 수 있을 것 같고, 조립도 간단해서 부모 입장에서도 편리했습니다. 가격대가 조금 있지만 그만한 가치가 충분한 제품입니다.', 'brand': '토이트론', 'title': '5096-실바니안 어린이 병원(2815)', 'category': '유아동', '@search.score': 0.6659234, '@search.reranker_score': None, '@search.highlights': None, '@se

1. [점수: 0.6605] [나이키] NSW 우븐 여성 MR 쇼츠 NIKE FV7558-010
   나이키 | 스포츠/레져 | 가격: 29,500원
   {'normal_price': 29500, 'review': '나이키 NSW 우븐 여성 MR 쇼츠는 가볍고 통기성이 좋아서 운동할 때 정말 편안했어요. 허리 밴드가 탄탄해서 흘러내리지 않고 잘 잡아줘서 활동하기 좋았습니다. 디자인도 심플하면서 세련돼서 일상복으로도 손색없고, 가격대비 품질이 뛰어나 만족스럽습니다. 여름철 가벼운 외출이나 운동복으로 추천합니다.', 'brand': '나이키', 'title': '[나이키] NSW 우븐 여성 MR 쇼츠 NIKE FV7558-010', 'category': '스포츠/레져', '@search.score': 0.66049623, '@search.reranker_score': None, '@search.highlights': None, '@search.captions': None}
2. [점수: 0.6519] [나이키] DF 피트니스 티셔츠 NIKE DX0990-010
   나이키 | 스포츠/레져 | 가격: 23,480원
   {'normal_price': 23480, 'review': '나이키 DF 피트니스 티셔츠는 가벼운 소재 덕분에 운동할 때 땀 흡수와 통기성이 정말 좋아요. 디자인도 심플하면서 세련돼서 일상복으로도 부담 없이 입기 좋았고, 착용감도 편안해서 장시간 착용해도 불편함이 없었어요. 가격 대비 품질도 만족스럽고 특히 뒷부분 메쉬 처리 덕분에 더 시원한 느낌이 들어 여름철 운동복으로 강추합니다.', 'brand': '나이키', 'title': '[나이키] DF 피트니스 티셔츠 NIKE DX0990-010', 'category': '스포츠/레져', '@search.score': 0.6519191, '@search.reranker_score': None, '@search.highlights': None, '@search.captions': None}
3. [점

1. [점수: 0.6460] [메이크업포에버][스벅 1잔] HD SKIN 퍼펙팅 프레스드 파우더 10g (+추가 2종)
   메이크업포에버 | 이미용 | 가격: 62,000원
   {'normal_price': 62000, 'review': '메이크업포에버 HD 스킨 프레스드 파우더는 피부 표현이 정말 자연스러워서 데일리 메이크업에 딱이에요. 유분기를 잘 잡아주면서도 건조하지 않고 가볍게 발려서 답답함이 전혀 없네요. 특히 미세한 입자가 모공이나 잔주름에 끼지 않고 매끈하게 마무리돼서 사진 찍을 때도 피부가 한층 깨끗해 보여 만족스러웠습니다. 추가로 받은 2종 샘플도 활용도가 높아 가성비도 좋았어요.', 'brand': '메이크업포에버', 'title': '[메이크업포에버][스벅 1잔] HD SKIN 퍼펙팅 프레스드 파우더 10g (+추가 2종)', 'category': '이미용', '@search.score': 0.6460158, '@search.reranker_score': None, '@search.highlights': None, '@search.captions': None}
2. [점수: 0.6391] [디올] 포에버 스킨 글로우 24H 웨어 래디언트 파운데이션
   크리스찬 디올 | 이미용 | 가격: 96,000원
   {'normal_price': 96000, 'review': '디올 포에버 스킨 글로우 파운데이션은 처음 써보는데 피부에 정말 자연스럽게 밀착되네요. 24시간 지속력이라고 해서 반신반의했지만 하루 종일 번들거림 없이 촉촉함이 유지돼서 만족스러웠어요. 커버력도 적당해서 잡티를 가려주면서도 두꺼워 보이지 않고, 광도 은은하게 나서 피부가 건강해 보입니다. 가격대는 있지만 그만큼 품질이 뛰어나고 데일리용으로 추천하고 싶어요.', 'brand': '크리스찬 디올', 'title': '[디올] 포에버 스킨 글로우 24H 웨어 래디언트 파운데이션', 'category': '이미용', '@search.score': 0.63912165, '@sea

### 💡 실습 결과 분석

**왜 두 검색이 거의 동일한가?**

실제로는 **둘 다 HNSW 알고리즘으로 검색**되었기 때문입니다:
- 같은 `content_vector` 필드 사용
- 이 필드는 `hnsw-profile`로 설정됨
- Azure AI Search에서 필드는 하나의 프로필만 가질 수 있음

**진짜 비교를 하려면?**

두 개의 별도 벡터 필드가 필요합니다:
```python
# 인덱스 정의 시
content_vector_exhaustive = SearchField(
    name="content_vector_exhaustive",
    vector_search_profile_name="knn-profile"  # Exhaustive
)

content_vector_hnsw = SearchField(
    name="content_vector_hnsw",
    vector_search_profile_name="hnsw-profile"  # HNSW
)
```

**핸즈온의 목적:**
- ✅ 두 알고리즘의 개념과 차이점 이해
- ✅ 벡터 검색 코드 작성법 학습
- ❌ 실제 성능 비교 (소규모 데이터 + 같은 알고리즘 사용)

**실무에서는:**
- 대부분 **HNSW만 사용**
- 수만~수백만 개 데이터에서 필수
- 정확도 손실은 거의 없음 (95-99%)
- Exhaustive는 특수한 경우에만 사용

---

## 5️⃣ 기본 벡터 검색 실습

이제 실제 검색 시나리오를 테스트해봅니다.

---

## 6️⃣ 키워드 검색 vs 벡터 검색 비교

같은 검색어로 두 방식을 비교해봅니다.

In [4]:
query_text = "가죽 가방"

print(f"\n{'='*80}")
print(f"🔍 비교 검색어: '{query_text}'")
print('='*80)

# 1. 키워드 검색 (BM25)
print("\n📝 1) 키워드 검색 (BM25)")
keyword_results = search_client.search(
    search_text=query_text,
    select=["title", "brand", "category", "normal_price", "review"],
    top=5
)

for idx, result in enumerate(keyword_results, 1):
    score = result.get('@search.score', 0)
    print(f"{idx}. [점수: {score:.4f}] {result['title']}")
    print(f"   {result['brand']} | {result['category']} | {result['normal_price']:,}원")
    print(f"   {result}\n")

# 2. 벡터 검색 (content_vector 사용)
print("\n🎯 2) 벡터 검색 (Semantic - content_vector)")
query_vector = get_embedding(query_text)
vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=5,
    fields="content_vector"  # 제품명 + 브랜드 + 카테고리 포함
)

vector_results = search_client.search(
    search_text=None,
    vector_queries=[vector_query],
    select=["title", "brand", "category", "normal_price", "review"],
    top=5
)

for idx, result in enumerate(vector_results, 1):
    score = result.get('@search.score', 0)
    print(f"{idx}. [점수: {score:.4f}] {result['title']}")
    print(f"   {result['brand']} | {result['category']} | {result['normal_price']:,}원\n")
    print(f"   {result}\n")


🔍 비교 검색어: '가죽 가방'

📝 1) 키워드 검색 (BM25)
1. [점수: 7.7741] 아가방 미키프렌즈손수건세트(10P) 79R0 85102
   아가방플렉스(백화점) | 유아동 | 22,000원
   {'normal_price': 22000, 'review': '아가방 미키프렌즈손수건세트는 디자인이 너무 귀여워서 아이가 좋아해요. 10장이 한 세트로 실용적이고, 부드러운 면 소재라 아기 피부에도 자극 없이 잘 맞습니다. 크기도 적당해서 외출 시 가방에 넣어 다니기 편하고, 세탁 후에도 쉽게 변형되지 않아 만족스러워요. 가격 대비 품질도 좋아서 선물용으로도 추천합니다.', 'brand': '아가방플렉스(백화점)', 'title': '아가방 미키프렌즈손수건세트(10P) 79R0 85102', 'category': '유아동', '@search.score': 7.774065, '@search.reranker_score': None, '@search.highlights': None, '@search.captions': None}

2. [점수: 6.7305] [메종 마르지엘라] 가죽 카드 지갑 T8013
   메종마르지엘라(OTB) | 패션잡화 | 670,000원
   {'normal_price': 670000, 'review': '메종 마르지엘라 가죽 카드 지갑은 고급스러운 가죽 질감이 정말 만족스러웠습니다. 슬림한 디자인 덕분에 주머니에 부담 없이 쏙 들어가고, 카드 수납 공간도 적당해서 실용적이에요. 특히 브랜드 특유의 미니멀함과 세련된 마감이 돋보여서 사용할 때마다 기분이 좋습니다. 가격대가 있는 편이지만 품질과 디자인 면에서 충분히 가치 있다고 생각해요. 매일 들고 다니기 좋은 지갑입니다.', 'brand': '메종마르지엘라(OTB)', 'title': '[메종 마르지엘라] 가죽 카드 지갑 T8013', 'category': '패션잡화', '@search.score': 6.730548, '@search.reranker_score': None, '

1. [점수: 0.6920] [메종 마르지엘라] 라지 가죽 체인 지갑 T8013
   메종마르지엘라(OTB) | 패션잡화 | 1,300,000원

   {'normal_price': 1300000, 'review': '메종 마르지엘라 라지 가죽 체인 지갑을 구매한 지 한 달쯤 됐는데 가죽 질감이 정말 부드러워서 사용감이 좋고 고급스러워 보여 만족스럽습니다. 체인 스트랩 덕분에 크로스로 멜 수 있어 손이 자유롭고, 내부 공간도 넉넉해 카드와 지폐, 간단한 소지품을 깔끔하게 정리할 수 있어 실용적이에요. 디자인이 심플하면서도 세련돼 어디에나 잘 어울려 데일리 아이템으로 제격입니다. 가격대가 부담스러울 수 있지만 품질과 디자인 모두 만족스러워 추천합니다.', 'brand': '메종마르지엘라(OTB)', 'title': '[메종 마르지엘라] 라지 가죽 체인 지갑 T8013', 'category': '패션잡화', '@search.score': 0.6919876, '@search.reranker_score': None, '@search.highlights': None, '@search.captions': None}

2. [점수: 0.6786] [메종 마르지엘라] 가죽 카드 지갑 T8013
   메종마르지엘라(OTB) | 패션잡화 | 670,000원

   {'normal_price': 670000, 'review': '메종 마르지엘라 가죽 카드 지갑은 고급스러운 가죽 질감이 정말 만족스러웠습니다. 슬림한 디자인 덕분에 주머니에 부담 없이 쏙 들어가고, 카드 수납 공간도 적당해서 실용적이에요. 특히 브랜드 특유의 미니멀함과 세련된 마감이 돋보여서 사용할 때마다 기분이 좋습니다. 가격대가 있는 편이지만 품질과 디자인 면에서 충분히 가치 있다고 생각해요. 매일 들고 다니기 좋은 지갑입니다.', 'brand': '메종마르지엘라(OTB)', 'title': '[메종 마르지엘라] 가죽 카드 지갑 T8013', 'category': '패션잡화', '@search.score': 0.

### 📊 비교 분석

**키워드 검색:**
- "가죽"과 "가방"이라는 **정확한 단어**가 있는 제품만 검색
- 점수는 단어 빈도와 희귀성(BM25)으로 계산
- 빠르고 정확하지만 유연성 부족

**벡터 검색:**
- "핸드백", "토트백", "크로스백" 같은 **유사 의미**도 검색
- 점수는 의미적 유사도(코사인 유사도)로 계산
- 더 자연스럽고 유연하지만 계산 비용이 높음

**언제 사용할까?**
- **키워드 검색**: 정확한 상품명, 브랜드명, 모델명 검색
- **벡터 검색**: 자연어 질문, 모호한 검색, 추천 시스템

---

## 7️⃣ 필터링 벡터 검색

벡터 검색에 필터 조건을 추가할 수 있습니다.

In [5]:
# 실용적인 필터 + 벡터 검색 시나리오
print("\n" + "="*80)
print("💡 필터 + 벡터 검색: 조건과 의미를 동시에 고려")
print("="*80)
print("사용 사례: 특정 카테고리와 가격대에서 의미 기반 검색\n")

# 시나리오 1: 예산 내에서 여름 옷 찾기
query_text = "여름에 입을 가벼운 옷"
category_filter = "패션의류"
max_price = 150000

print(f"🔍 검색어: '{query_text}'")
print(f"🎯 필터 조건:")
print(f"   - 카테고리: {category_filter}")
print(f"   - 최대 가격: {max_price:,}원 이하")
print(f"\n📝 벡터 필드: content_vector (제품명 + 브랜드 + 카테고리)\n")

# 벡터화
query_vector = get_embedding(query_text)

# 벡터 쿼리 - k를 크게 설정하여 필터링 후에도 충분한 결과 확보
vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=50,  # 필터 전에 더 많이 가져오기 (중요!)
    fields="content_vector"
)

# 필터 조건 (OData 형식)
filter_expression = f"category eq '{category_filter}' and normal_price le {max_price}"

# 검색 실행
results = search_client.search(
    search_text=None,
    vector_queries=[vector_query],
    filter=filter_expression,
    select=["title", "brand", "category", "normal_price"],
    top=10
)

result_list = list(results)
print(f"{'='*80}")
print(f"검색 결과: {len(result_list)}개 제품 발견")
print('='*80)

print(f"\n💡 필터 + 벡터의 장점:")
print(f"   ✅ 카테고리 필터로 관련 없는 제품 제외")
print(f"   ✅ 가격 필터로 예산 범위 내 검색")
print(f"   ✅ 벡터 검색으로 '여름', '가벼운' 같은 의미 반영")
print(f"   ✅ 세 가지 조건을 모두 만족하는 최적의 결과")

for idx, result in enumerate(result_list, 1):
    score = result.get('@search.score', 0)
    print(f"\n{idx}. [점수: {score:.4f}] {result['title']}")
    print(f"   {result['brand']} | {result['category']}")
    print(f"   가격: {result['normal_price']:,}원")



💡 필터 + 벡터 검색: 조건과 의미를 동시에 고려
사용 사례: 특정 카테고리와 가격대에서 의미 기반 검색

🔍 검색어: '여름에 입을 가벼운 옷'
🎯 필터 조건:
   - 카테고리: 패션의류
   - 최대 가격: 150,000원 이하

📝 벡터 필드: content_vector (제품명 + 브랜드 + 카테고리)



검색 결과: 4개 제품 발견

💡 필터 + 벡터의 장점:
   ✅ 카테고리 필터로 관련 없는 제품 제외
   ✅ 가격 필터로 예산 범위 내 검색
   ✅ 벡터 검색으로 '여름', '가벼운' 같은 의미 반영
   ✅ 세 가지 조건을 모두 만족하는 최적의 결과

1. [점수: 0.6039] 앤더슨벨 커트 스트라이프 롱슬리브 티셔츠 atb1247m(ECRU)
   앤더슨벨 | 패션의류
   가격: 103,500원

2. [점수: 0.5953] 라코스테우먼 25SS 여성 베이직 린넨셔츠 CF905E-55G AFS
   라코스테 우먼 | 패션의류
   가격: 137,970원

3. [점수: 0.5908] 라코스테우먼 25SS 여성 스트라이프 긴팔 티셔츠 TF922E-55G HBP
   라코스테 우먼 | 패션의류
   가격: 93,870원

4. [점수: 0.5895] 라코스테우먼 25FW 례귤러핏 폴로셔츠 PF7839-55N 02S
   라코스테 우먼 | 패션의류
   가격: 127,200원


---

## 8️⃣ 멀티 벡터 검색 (Multi-Vector Search)

여러 벡터 필드를 동시에 검색할 수 있습니다.

### 사용 사례
- **content_vector**: 제품명, 브랜드, 카테고리를 포함한 종합 검색
- **review_vector**: 리뷰 기반 검색
- 두 필드를 동시에 검색하여 더 정확한 결과 도출

In [6]:
query_text = "배송 빠르고 품질 좋은 제품"

print(f"\n{'='*80}")
print(f"🔍 멀티 벡터 검색: '{query_text}'")
print('='*80)

# 검색어 벡터화
query_vector = get_embedding(query_text)

# 벡터 쿼리 1: content_vector (제품명 + 브랜드 + 카테고리)
content_vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=10,
    fields="content_vector"
)

# 벡터 쿼리 2: review_vector
review_vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=10,
    fields="review_vector"
)

# 멀티 벡터 검색 실행
results = search_client.search(
    search_text=None,
    vector_queries=[content_vector_query, review_vector_query],  # 두 벡터 쿼리 사용
    select=["title", "brand", "category", "normal_price", "review"],
    top=10
)

print_results(results, title="🎯 멀티 벡터 검색 결과 (content + review)")

print("\n💡 멀티 벡터 검색의 장점:")
print("   - content_vector: 제품 정보 (제품명 + 브랜드 + 카테고리) 검색")
print("   - review_vector: 리뷰 내용 검색")
print("   - '배송 빠르고 품질 좋은'은 주로 리뷰에 등장")
print("   - 두 벡터를 결합하여 더 포괄적이고 정확한 검색 결과")


🔍 멀티 벡터 검색: '배송 빠르고 품질 좋은 제품'



🎯 멀티 벡터 검색 결과 (content + review)



1. [점수: 0.0325] 사브르 비스트로 샤이니 샐러드 선물세트
   브랜드: 사브르 | 카테고리: 주방
   가격: 47,120원
   리뷰: 사브르 비스트로 샤이니 샐러드 선물세트는 신선한 재료와 고급스러운 포장 덕분에 선물용으로 딱 좋았어요. 샐러드 채소가 아삭아삭하고 드레싱도 적당히 상큼해서 식사할 때 부담 없이 즐...

2. [점수: 0.0315] 사브르 비스트로 샤이니 디저트 선물세트
   브랜드: 사브르 | 카테고리: 주방
   가격: 41,800원
   리뷰: 사브르 비스트로 샤이니 디저트 선물세트를 부모님 생신 선물로 구매했는데 포장부터 고급스러워서 만족스러웠어요. 각 디저트가 하나하나 정성스럽게 만들어져 있어서 맛도 다양하고 풍부했습...

3. [점수: 0.0167] [킬리안] 굿 걸 곤 배드 50ml (리필 가능)
   브랜드: 킬리안-향수 | 카테고리: 이미용
   가격: 378,000원
   리뷰: 킬리안 굿 걸 곤 배드는 처음 맡았을 때부터 고급스러운 플로럴 향과 우디한 느낌이 잘 어우러져 매우 만족스러웠습니다. 50ml 용량에 리필이 가능해 환경을 생각하는 점도 좋고, 지...

4. [점수: 0.0167] [어니스트서울] 14K 랩다이아몬드 5부 4프롱 목걸이
   브랜드: 어니스트서울 | 카테고리: 보석/장신구
   가격: 969,000원
   리뷰: 어니스트서울 14K 랩다이아몬드 목걸이를 선물용으로 구매했는데 디자인이 정말 세련되고 고급스러워서 만족스러웠어요. 5부 다이아몬드가 반짝임이 뛰어나고 4프롱 세팅 덕분에 안정감도 ...

5. [점수: 0.0164] [디즈니 스토어 공식 상품] 픽사 토이스토리 버즈 라이트이어 베이비 코스튬 (104Q419D133)
   브랜드: 디즈니 | 카테고리: 유아동
   가격: 41,400원
   리뷰: 아이 생일 파티 때 입히려고 구매했는데 디자인이 정말 귀엽고 퀄리티가 좋아서 만족스러웠어요. 부드러운 재질이라 아이가 피부가 예민한데도 문제없이 편안해했고, 착용도 간편해서 도움 .

---

## 9️⃣ 자연어 질문 검색

사용자가 자연스러운 문장으로 검색하는 시나리오입니다.

벡터 검색은 자연어 질문에 강력합니다!

In [7]:
# 예제 1: 자연어 질문
query_text = "지성 피부를 위한 화장품"

print(f"\n{'='*80}")
print(f"🔍 자연어 질문: '{query_text}'")
print('='*80)

query_vector = get_embedding(query_text)
vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=5,
    #fields="content_vector"
    fields="review_vector"
)

results = search_client.search(
    search_text=None,
    vector_queries=[vector_query],
    select=["title", "brand", "category", "normal_price","review"],
    top=5
)

for idx, result in enumerate(results, 1):
    score = result.get('@search.score', 0)
    print(f"{idx}. [점수: {score:.4f}] {result['title']}")
    print(f"   {result['brand']} | {result['category']}")
    print(f"   {result}\n")


🔍 자연어 질문: '지성 피부를 위한 화장품'


1. [점수: 0.6279] 설화수[8월]윤조에센스 6세대 90ml 기획세트
   설화수 | 이미용
   {'normal_price': 140000, 'review': '평소 건조하고 예민한 피부 때문에 고민이 많았는데 설화수 윤조에센스 6세대 사용 후 확실히 피부가 촉촉해지고 탄력이 생겼어요. 90ml 대용량이라 오래 쓸 수 있어 가성비도 좋고, 흡수도 빨라서 아침저녁으로 부담 없이 바르기 좋아요. 기획세트라 추가로 샘플도 받아 다양한 제품을 체험해볼 수 있어 만족스럽습니다. 꾸준히 사용하면 피부톤이 한층 밝아지는 느낌이라 재구매 의사 있습니다.', 'brand': '설화수', 'title': '설화수[8월]윤조에센스 6세대 90ml 기획세트', 'category': '이미용', '@search.score': 0.62792593, '@search.reranker_score': None, '@search.highlights': None, '@search.captions': None}

2. [점수: 0.6258] [선쿠션증정][보보쇼즈 X 노베즈 콜라보] For all senses 포 올 센스 (로션+올인원워시+페이스크림)
   노베즈 | 유아동
   {'normal_price': 99000, 'review': '아이 피부가 예민해서 보통 제품은 잘 못 쓰는데 노베즈 콜라보 제품은 순하고 자극 없어 안심하고 사용하고 있어요. 로션은 촉촉하면서도 끈적임 없이 산뜻하고 올인원 워시는 거품이 부드러워 아이 목욕 시간마다 거부감 없이 잘 사용해요. 페이스크림도 적당한 보습감으로 하루종일 피부가 편안해 보입니다. 선쿠션 증정 이벤트 덕분에 더 실속 있게 구매했네요. 전체적으로 아이 피부를 세심하게 케어해주는 느낌이라 만족도가 높습니다.', 'brand': '노베즈', 'title': '[선쿠션증정][보보쇼즈 X 노베즈 콜라보] For all senses 포 올 센스 (로션+올인원워시+페이스크림)', 'category': '유아동', '@search.score': 0.6

In [8]:
# 예제 2: 상황 설명
query_text = "간지러운 두피가 고민이에요"

print(f"\n{'='*80}")
print(f"🔍 상황 설명: '{query_text}'")
print('='*80)

query_vector = get_embedding(query_text)
vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=5,
    #fields="content_vector"
    fields="review_vector"
)

results = search_client.search(
    search_text=None,
    vector_queries=[vector_query],
    select=["title", "brand", "category", "normal_price","review"],
    top=5
)

for idx, result in enumerate(results, 1):
    score = result.get('@search.score', 0)
    print(f"{idx}. [점수: {score:.4f}] {result['title']}")
    print(f"   {result['brand']} | {result['category']}")
    print(f"   {result}\n")


🔍 상황 설명: '간지러운 두피가 고민이에요'


1. [점수: 0.6099] 독샤워 리커버리인아쿠아(리아) 샴푸
   독샤워 | 문화/취미
   {'normal_price': 46000, 'review': '독샤워 리커버리인아쿠아 샴푸를 몇 주째 사용 중인데 모발이 한층 부드러워지고 윤기가 돌아서 만족스러워요. 특히 건조하고 손상된 머릿결에 수분을 잘 공급해주는 느낌이라 겨울철에도 잘 맞는 제품 같아요. 향도 은은해서 부담 없이 매일 사용할 수 있고, 두피가 가렵거나 자극받는 느낌도 없어서 민감한 분들에게도 추천합니다. 가격대가 조금 있지만 효과가 좋아서 재구매 의사 있습니다.', 'brand': '독샤워', 'title': '독샤워 리커버리인아쿠아(리아) 샴푸', 'category': '문화/취미', '@search.score': 0.6099362, '@search.reranker_score': None, '@search.highlights': None, '@search.captions': None}

2. [점수: 0.6081] [르네휘테르]포티샤 두피 모발 마스크 600ml(+200ml 증정)
   르네휘테르 | 이미용
   {'normal_price': 66000, 'review': '르네휘테르 포티샤 두피 모발 마스크는 두피가 예민한 편인데도 자극 없이 편안하게 사용할 수 있었어요. 무겁거나 끈적이지 않고 부드럽게 발리면서 모발에 영양을 꽉 채워주는 느낌이 듭니다. 사용 후에는 머릿결이 한결 매끄럽고 건강해진 듯하며, 두피도 한층 깔끔해진 기분이에요. 대용량에 증정까지 포함되어 가성비도 만족스럽고 꾸준히 쓰기 좋은 제품입니다.', 'brand': '르네휘테르', 'title': '[르네휘테르]포티샤 두피 모발 마스크 600ml(+200ml 증정)', 'category': '이미용', '@search.score': 0.60809326, '@search.reranker_score': None, '@search.highlights': None, '@search.captions': None}

3. [점수

---

## 🔟 추천 시스템 구현

특정 제품과 유사한 제품을 찾는 "이 제품과 유사한 상품" 기능을 구현합니다.

In [9]:
# 특정 제품 선택
target_product_id = "10"

print(f"\n{'='*80}")
print(f"💡 추천 시스템: '이 상품과 유사한 제품' 구현")
print('='*80)
print("벡터 검색을 활용한 상품 추천 시스템입니다.")
print("기준 상품의 텍스트를 임베딩하여 유사한 제품을 찾습니다.\n")

# 제품 정보 가져오기
target_doc = search_client.get_document(
    key=target_product_id,
    selected_fields=["id", "title", "brand", "category", "normal_price"]
)

print(f"{'='*80}")
print(f"📦 기준 제품 (ID: {target_product_id})")
print('='*80)
print(f"제품명: {target_doc['title']}")
print(f"브랜드: {target_doc['brand']}")
print(f"카테고리: {target_doc['category']}")
print(f"가격: {target_doc['normal_price']:,}원")

# content_vector는 retrievable=False이므로 직접 조회 불가
# 대신 제품 텍스트로 임베딩을 생성하여 유사 제품 검색
target_text = f"{target_doc['title']} {target_doc['brand']} {target_doc['category']}"
target_vector = get_embedding(target_text)
print(f"\n✅ 제품 텍스트로 임베딩 생성: '{target_text}'")
print(f"   벡터 차원: {len(target_vector)}차원")

# 이 제품의 벡터로 유사 제품 검색
vector_query = VectorizedQuery(
    vector=target_vector,
    k_nearest_neighbors=6,  # 자기자신 포함 6개
    fields="content_vector"
)

results = search_client.search(
    search_text=None,
    vector_queries=[vector_query],
    select=["id", "title", "brand", "category", "normal_price"],
    top=6
)

print(f"\n{'='*80}")
print(f"🎯 유사한 제품 추천 (벡터 유사도 기반)")
print('='*80)

recommendation_count = 0
for idx, result in enumerate(results, 1):
    if result['id'] == target_product_id:
        continue  # 자기 자신은 제외
    
    recommendation_count += 1
    score = result.get('@search.score', 0)
    print(f"\n{recommendation_count}. [유사도: {score:.4f}] {result['title']}")
    print(f"   {result['brand']} | {result['category']} | {result['normal_price']:,}원")

print(f"\n{'='*80}")
print(f"💡 추천 시스템의 활용:")
print('='*80)
print(f"   - 전자상거래: '이 상품과 유사한 제품' 섹션")
print(f"   - 크로스셀링: 관련 제품 추천으로 매출 증대")
print(f"   - 개인화: 사용자 관심 제품 기반 추천")
print(f"   - content_vector로 제품명, 브랜드, 카테고리를 모두 고려한 추천")
print(f"\n💡 참고:")
print(f"   - content_vector는 retrievable=False이므로 직접 가져올 수 없음")
print(f"   - 대신 제품 텍스트로 임베딩을 새로 생성하여 검색")
print(f"   - 실무에서는 retrievable=True로 설정하면 저장된 벡터를 재사용 가능")


💡 추천 시스템: '이 상품과 유사한 제품' 구현
벡터 검색을 활용한 상품 추천 시스템입니다.
기준 상품의 텍스트를 임베딩하여 유사한 제품을 찾습니다.

📦 기준 제품 (ID: 10)
제품명: [M밍크뮤D] (WH)루루뱀부손수건SET 10장세트 (3417000603) 출산준비물/신생아선물/배내수트/배내저고리
브랜드: 밍크뮤
카테고리: 유아동
가격: 20,880원



✅ 제품 텍스트로 임베딩 생성: '[M밍크뮤D] (WH)루루뱀부손수건SET 10장세트 (3417000603) 출산준비물/신생아선물/배내수트/배내저고리 밍크뮤 유아동'
   벡터 차원: 3072차원

🎯 유사한 제품 추천 (벡터 유사도 기반)

1. [유사도: 0.7796] [M밍크뮤M7]35970BHD02 드림루루딸랑이세트/유아용품/출산준비/임신/출산/백일/조카선물
   밍크뮤 | 유아동 | 35,280원

2. [유사도: 0.7606] [M밍크뮤D] 26년말띠(WH)드림코니리오셀 배내저고리양말손싸개세트 (35A700110301) 출산준비물/신생아선물/
   밍크뮤 | 유아동 | 77,600원

3. [유사도: 0.7472] 블루독베이비[hu] WH)블루독남아손수건SET 41170-006-01 손수건/타올/수건/10장세트/신생아/출산용품/선물
   블루독베이비 | 유아동 | 22,190원

4. [유사도: 0.7442] 블루독베이비[hu] PK)블루독뱀부여아손수건SET 43170-006-03 손수건/타올/수건/10장/신생아/출산용품/선물
   블루독베이비 | 유아동 | 22,190원

5. [유사도: 0.7186] [B블루독베이비B] (WH)도기딸랑이손수건SET    44970BHD02  선물/인형
   블루독베이비 | 유아동 | 36,000원

💡 추천 시스템의 활용:
   - 전자상거래: '이 상품과 유사한 제품' 섹션
   - 크로스셀링: 관련 제품 추천으로 매출 증대
   - 개인화: 사용자 관심 제품 기반 추천
   - content_vector로 제품명, 브랜드, 카테고리를 모두 고려한 추천

💡 참고:
   - content_vector는 retrievable=False이므로 직접 가져올 수 없음
   - 대신 제품 텍스트로 임베딩을 새로 생성하여 검색
   - 실무에서는 retrievable=True로 설정하면 저장된 벡터를 재사용 가능


---

## ✅ 학습 완료!

축하합니다! 벡터 검색의 핵심 기능을 모두 학습했습니다.

### 📚 학습한 내용

1. ✅ **기본 벡터 검색**: 의미 기반 검색
2. ✅ **키워드 vs 벡터**: 두 방식의 차이점 이해
3. ✅ **하이브리드 검색**: 키워드 + 벡터 결합
4. ✅ **필터링 벡터 검색**: 조건과 벡터 검색 결합
5. ✅ **멀티 벡터 검색**: 여러 필드 동시 검색
6. ✅ **자연어 질문**: 자연스러운 문장으로 검색
7. ✅ **추천 시스템**: 유사 제품 추천

### 🎯 벡터 검색의 핵심 장점

| 장점 | 설명 | 예시 |
|------|------|------|
| **의미 이해** | 키워드가 없어도 의미만 유사하면 검색 | "노트북" → "랩탑" |
| **자연어 지원** | 자연스러운 문장으로 검색 가능 | "비올 때 신을 신발" |
| **다국어** | 언어가 달라도 의미가 같으면 검색 | "laptop" → "노트북" |
| **추천** | 유사한 제품 자동 추천 | "이 상품과 유사한 제품" |

### 🚀 다음 단계

이제 더 고급 검색 기능을 학습할 준비가 되었습니다:

1. **Semantic Search** - AI 재순위화로 더욱 정확한 검색
2. **Scoring Profiles** - 비즈니스 로직 기반 점수 조정
3. **AI Skills** - 이미지, 텍스트 분석 통합

### 📖 참고 자료

- [Azure AI Search Vector Search](https://learn.microsoft.com/azure/search/vector-search-overview)
- [HNSW Algorithm](https://arxiv.org/abs/1603.09320)
- [Hybrid Search Best Practices](https://learn.microsoft.com/azure/search/hybrid-search-overview)

---

## 🧭 다음 단계

| ⬅️ 이전 | 🏠 목차 | ➡️ 다음 |
|:---------|:-------:|--------:|
| [Lab 03-2: 벡터 업로드](02-upload_vectors.ipynb) | [워크숍 홈](../README.md) | [Lab 04: 하이브리드 검색](../04-hybrid_search/01-hybrid_search.ipynb) |